# f6_m02a_lime.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M02a — LIME |

---

## 🎯 Qué hace

Genera explicaciones locales con LIME (Local Interpretable Model-agnostic Explanations)
sobre los 3 perfiles representativos definidos en m01b (abandono, no abandono, riesgo).
LIME complementa SHAP: mientras SHAP usa valores de Shapley exactos basados en el modelo,
LIME aproxima localmente el comportamiento con un modelo lineal simple.
Modelo usado: CatBoost__balanced.

## 📋 Requisitos

- `results/fase6/perfiles_locales.parquet` — generado por f6_m01b
- `results/fase6/shap_importancia_comparativa.parquet` — generado por f6_m01a
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_train_prep.parquet` — referencia de distribución para LIME
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`
- Paquete: lime

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/lime_abandono_real.png` | Gráfico LIME perfil abandono |
| `results/fase6/lime_no_abandono.png` | Gráfico LIME perfil no abandono |
| `results/fase6/lime_zona_riesgo.png` | Gráfico LIME perfil zona riesgo |
| `results/fase6/lime_explicaciones.parquet` | Coeficientes LIME para los 3 perfiles |
| `docs/html/fase6/m02a_lime.html` | Informe HTML |

## 🔄 Flujo

```
perfiles_locales.parquet (m01b) + X_train_prep + CatBoost__balanced.pkl
    ↓ Instanciar LimeTabularExplainer con datos de entrenamiento
    ↓ Explicar 3 perfiles (5000 perturbaciones cada uno)
    ↓ Gráficos de barras por perfil
    ↓ Comparativa consenso SHAP vs LIME
    → lime_explicaciones.parquet + docs/html/fase6/m02a_lime.html
```

## ➡️ Siguiente

`f6_m02b_dice.ipynb` — contrafactuales DiCE


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

print(f'ROOT:       {ROOT}')
print(f'DIR_MODELS: {DIR_MODELS}')
print(f'DIR_RESULTS:{DIR_RESULTS}')

ROOT:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_MODELS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\05_modelado\models
DIR_RESULTS:C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# ============================================================
# CELDA 2: IMPORTS
# ============================================================
import base64
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from lime import lime_tabular
from src.html.render import render_pagina_desde_fichero

matplotlib.rcParams['figure.dpi'] = 120

print('Imports OK.')
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES

def nombre_legible(f: str) -> str:
    return NOMBRES_LEGIBLES_FEATURES.get(f, f.replace("_", " "))


Imports OK.


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y MODELO
# X_train_prep: LIME necesita datos de entrenamiento como
# referencia de distribución para generar perturbaciones.
# perfiles_locales: los 3 perfiles definidos en m01b.
# ============================================================
RUTA_PERFILES = DIR_RESULTS / 'perfiles_locales.parquet'
if not RUTA_PERFILES.exists():
    raise FileNotFoundError(
        'No se encontró perfiles_locales.parquet.\n'
        'Ejecuta primero f6_m01b_shap_local.ipynb.'
    )

X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
X_train_prep = pd.read_parquet(DIR_DATA / 'X_train_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
df_perfiles  = pd.read_parquet(RUTA_PERFILES)
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

feature_names = X_test_prep.columns.tolist()

# Detectar features categóricas — LIME necesita saberlo
cat_features = X_test_prep.select_dtypes(include=['object', 'category']).columns.tolist()
cat_idx      = [feature_names.index(f) for f in cat_features]

print(f'X_test_prep:  {X_test_prep.shape}')
print(f'X_train_prep: {X_train_prep.shape}')
print(f'Features:     {len(feature_names)} ({len(cat_features)} categóricas)')
print(f'Perfiles:     {df_perfiles["perfil"].tolist()}')

X_test_prep:  (6725, 27)
X_train_prep: (26896, 27)
Features:     27 (0 categóricas)
Perfiles:     ['abandono_real', 'no_abandono', 'zona_riesgo']


In [4]:
# ============================================================
# CELDA 4: INSTANCIAR LIME EXPLAINER
# LimeTabularExplainer aprende la distribución de cada feature
# desde X_train para generar perturbaciones realistas.
# discretize_continuous=True: discretiza numéricas en intervalos
# para que el modelo lineal local sea más interpretable.
# ============================================================
explainer_lime = lime_tabular.LimeTabularExplainer(
    training_data=X_train_prep.values,
    feature_names=feature_names,
    categorical_features=cat_idx,
    class_names=['No abandona', 'Abandona'],
    mode='classification',
    discretize_continuous=True,
    random_state=42
)

print('✅ LIME explainer instanciado.')
print(f'   Referencia: {X_train_prep.shape[0]:,} observaciones de entrenamiento')

✅ LIME explainer instanciado.
   Referencia: 26,896 observaciones de entrenamiento


In [5]:
# ============================================================
# CELDA 5: GENERAR EXPLICACIONES LIME — 3 PERFILES
# 5000 perturbaciones por perfil — estándar LIME.
# Recupera índices posicionales desde perfiles_locales.
# ============================================================
prob  = pipeline_cat.predict_proba(X_test_prep)[:, 1]
y_arr = y_test.values.ravel()

# Buscar perfil trabajador — alumno que trabaja y abandona
# (n_anios_trabajando > 0 en X_test_prep)
col_trabaja = 'n_anios_trabajando' if 'n_anios_trabajando' in X_test_prep.columns else None
if col_trabaja:
    mask_trab = (X_test_prep[col_trabaja] > 0) & (y_arr == 1)
    if mask_trab.sum() > 0:
        idx_trab_pos = int(np.where(mask_trab)[0][0])
        df_perfiles_ext = pd.concat([
            df_perfiles,
            X_test_prep.iloc[[idx_trab_pos]].rename(index={X_test_prep.index[idx_trab_pos]: 'trabajador_abandona'})
        ])
        idx_originales   = df_perfiles_ext.index.tolist()
        idx_posicionales = [X_test_prep.index.get_loc(idx) if idx in X_test_prep.index
                            else idx_trab_pos for idx in idx_originales]
        print(f'Perfil trabajador añadido: idx_pos={idx_trab_pos}, prob={prob[idx_trab_pos]:.3f}')
    else:
        idx_posicionales = [X_test_prep.index.get_loc(idx) for idx in df_perfiles.index.tolist()]
else:
    idx_posicionales = [X_test_prep.index.get_loc(idx) for idx in df_perfiles.index.tolist()]

perfiles_orden  = ['abandono_real', 'no_abandono', 'zona_riesgo', 'trabajador_abandona'] if col_trabaja and mask_trab.sum() > 0 else ['abandono_real', 'no_abandono', 'zona_riesgo']
perfiles_labels = {
    'abandono_real': 'Abandono (VP)',
    'no_abandono':   'No abandona (VN)',
    'zona_riesgo':          'Zona de riesgo',
    'trabajador_abandona':  'Trabajador que abandona (4º perfil)',
}

explicaciones = {}
registros     = []

for perfil, idx_pos in zip(perfiles_orden, idx_posicionales):
    label    = perfiles_labels[perfil]
    instancia = X_test_prep.iloc[idx_pos].values
    print(f'Calculando LIME para {label}...')

    exp = explainer_lime.explain_instance(
        data_row=instancia,
        predict_fn=pipeline_cat.predict_proba,
        num_features=15,
        num_samples=5000,
        labels=(1,)
    )
    explicaciones[perfil] = exp

    for feat, coef in exp.as_list(label=1):
        registros.append({
            'perfil':    perfil,
            'feature':   feat,
            'coef_lime': coef,
            'prob_real': prob[idx_pos],
            'y_real':    y_arr[idx_pos],
        })
    print(f'  ✅ Prob: {prob[idx_pos]:.3f} | y_real: {y_arr[idx_pos]}')

df_lime = pd.DataFrame(registros)
df_lime.to_parquet(DIR_RESULTS / 'lime_explicaciones.parquet')
print(f'\n✅ lime_explicaciones.parquet guardado.')

Calculando LIME para Abandono (VP)...
  ✅ Prob: 0.992 | y_real: 1
Calculando LIME para No abandona (VN)...
  ✅ Prob: 0.051 | y_real: 0
Calculando LIME para Zona de riesgo...
  ✅ Prob: 0.500 | y_real: 1

✅ lime_explicaciones.parquet guardado.


In [6]:
# ============================================================
# CELDA 6: GRÁFICOS LIME — UNO POR PERFIL
# Barras horizontales:
#   rojo/naranja = empuja hacia abandono (coef positivo)
#   azul = reduce el riesgo (coef negativo)
# ============================================================
rutas_lime = {}

for perfil, idx_pos in zip(perfiles_orden, idx_posicionales):
    label = perfiles_labels[perfil]
    exp   = explicaciones[perfil]
    ruta  = DIR_RESULTS / f'lime_{perfil}.png'
    rutas_lime[perfil] = ruta

    feats_coefs = exp.as_list(label=1)
    feats = [fc[0] for fc in feats_coefs]
    coefs = [fc[1] for fc in feats_coefs]

    # Ordenar por valor absoluto
    orden = np.argsort(np.abs(coefs))
    feats = [feats[j] for j in orden]
    coefs = [coefs[j] for j in orden]

    colores_barras = ['#e53e3e' if c > 0 else '#3182ce' for c in coefs]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feats, coefs, color=colores_barras, alpha=0.8)
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_xlabel('Coeficiente LIME (positivo = hacia abandono)')
    ax.set_title(f'Fase 6 — LIME: {label} (prob={prob[idx_pos]:.3f})', fontsize=13, pad=12)
    plt.tight_layout()
    plt.savefig(ruta, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'✅ Gráfico guardado: {ruta.name}')

✅ Gráfico guardado: lime_abandono_real.png
✅ Gráfico guardado: lime_no_abandono.png
✅ Gráfico guardado: lime_zona_riesgo.png


In [7]:
# ============================================================
# CELDA 7: COMPARATIVA SHAP vs LIME — VISUAL Y TEXTUAL 🏆
# Features en consenso = hallazgos más robustos.
# Gráfico de barras coloreado: verde=consenso, gris=solo SHAP,
# naranja=solo LIME. Tabla HTML con semáforo de colores.
# ============================================================
RUTA_SHAP_IMP = DIR_RESULTS / 'shap_importancia_comparativa.parquet'
consenso = []
ruta_consenso = None

if RUTA_SHAP_IMP.exists():
    df_shap_imp = pd.read_parquet(RUTA_SHAP_IMP)
    top10_shap  = df_shap_imp.nsmallest(10, 'rank_medio').index.tolist()

    lime_abandono = df_lime[df_lime['perfil'] == 'abandono_real'].copy()
    lime_abandono['feature_base'] = lime_abandono['feature'].str.extract(r'^([^<>=!]+)')[0].str.strip()
    top10_lime = lime_abandono.nlargest(10, 'coef_lime')['feature_base'].tolist()

    # Detectar consenso: feature SHAP que aparece en algún label LIME
    consenso = [f for f in top10_shap if any(f in fl for fl in top10_lime)]

    print(f'Top 10 SHAP | Top 10 LIME | Consenso: {len(consenso)}/10')
    for f in top10_shap:
        marca = '✅ CONSENSO' if f in consenso else '  solo SHAP'
        print(f'  {nombre_legible(f):35s} {marca}')

    # Gráfico visual de consenso
    shap_vals = df_shap_imp.loc[top10_shap, 'CatBoost'].values
    colores   = ['#38a169' if f in consenso else '#a0aec0' for f in top10_shap]
    labels    = [nombre_legible(f) for f in top10_shap]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(labels[::-1], shap_vals[::-1], color=colores[::-1], alpha=0.85)
    ax.set_xlabel('Importancia SHAP media (CatBoost)')
    ax.set_title(
        'Fase 6 — Consenso SHAP + LIME: Top 10 features\n'        'Verde = aparece también en LIME (hallazgo robusto) · Gris = solo SHAP',
        fontsize=12
    )
    # Leyenda manual
    from matplotlib.patches import Patch
    ax.legend(handles=[
        Patch(color='#38a169', label=f'Consenso SHAP+LIME ({len(consenso)} features)'),
        Patch(color='#a0aec0', label=f'Solo SHAP ({10-len(consenso)} features)'),
    ], fontsize=9, loc='lower right')
    plt.tight_layout()
    ruta_consenso = DIR_RESULTS / 'lime_consenso_shap.png'
    plt.savefig(ruta_consenso, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'\n✅ Gráfico consenso guardado: {ruta_consenso.name}')
else:
    consenso = []
    top10_shap = []
    print('shap_importancia_comparativa.parquet no encontrado.')


Top 10 SHAP (consenso tres modelos):
  cred_superados_anio_1er             ✅ consenso SHAP+LIME
  n_anios_beca                        ✅ consenso SHAP+LIME
  n_anios_trabajando                  ✅ consenso SHAP+LIME
  n_anios_sin_notas                   
  nota_1er_anio                       ✅ consenso SHAP+LIME
  cred_repetidos                      
  anios_sin_beca                      
  situacion_laboral                   
  tasa_repeticion                     
  nota_acceso                         ✅ consenso SHAP+LIME

Features en consenso SHAP+LIME: 5/10


In [8]:
# ============================================================
# CELDA 8: GENERAR HTML
# render_pagina_desde_fichero — estándar del proyecto.
# ============================================================
def img_b64(ruta: Path) -> str:
    if not ruta.exists():
        return ''
    with open(ruta, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:#e53e3e">⚠️ Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

contenido = (
    '<h2 style="color:#2d3748">Fase 6 — LIME: Explicaciones Locales Alternativas</h2>'
    '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    'LIME (<em>Local Interpretable Model-agnostic Explanations</em>) aproxima el comportamiento '
    'del modelo localmente con un modelo lineal simple. A diferencia de SHAP, que usa valores '
    'de Shapley exactos, LIME genera perturbaciones de la instancia y ajusta un modelo lineal '
    'en esa región. Ambos métodos son complementarios: el consenso entre SHAP y LIME '
    'identifica las variables más robustamente importantes.'
    '</p>'
    + bloque_imagen(
        img_b64(rutas_lime['abandono_real']),
        'LIME — Alumno que abandona (verdadero positivo)',
        'Barras rojas: factores que el modelo lineal local identifica como impulsores del abandono. '
        'Barras azules: factores protectores. '
        'Los valores son coeficientes del modelo lineal local ajustado por LIME.')
    + bloque_imagen(
        img_b64(rutas_lime['no_abandono']),
        'LIME — Alumno que no abandona (verdadero negativo)',
        'Las barras azules dominan: LIME identifica factores protectores '
        'que reducen el riesgo predicho.')
    + bloque_imagen(
        img_b64(rutas_lime['zona_riesgo']),
        'LIME — Alumno en zona de riesgo (probabilidad ~0.5)',
        'Factores de riesgo y protectores se equilibran. '
        'Este perfil es el más sensible a pequeños cambios en las variables.')
    + (bloque_imagen(
        img_b64(ruta_consenso) if ruta_consenso else '',
        '🏆 Consenso SHAP + LIME — Features más robustas',
        'Verde = features que aparecen en el top 10 de SHAP Y en las más importantes de LIME. '
        'Estas son las variables más robustamente importantes del modelo: '
        'dos métodos completamente distintos llegan a la misma conclusión.'
    ) if ruta_consenso else '')
    + '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
    'border-left:4px solid #3182ce;border-radius:6px;font-size:13px;color:#2c5282">'
    '<strong>SHAP vs LIME:</strong> SHAP calcula contribuciones exactas basadas en la '
    'teoría de juegos (valores de Shapley). LIME aproxima localmente con un modelo lineal. '
    'Cuando ambos métodos coinciden en las variables más importantes, '
    'el hallazgo es especialmente robusto y defendible ante el tribunal.'
    '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m02a_lime.ipynb', contenido)
ruta_html = DIR_HTML / 'm02a_lime.html'
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m02a_lime.html
